Bronze Layer

In [0]:
import os
import requests

RAW_DIR = "/Volumes/workspace/src/cfpb/raw"
ZIP_PATH = f"{RAW_DIR}/complaints.csv.zip"
URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1e9:
    print(f"Already downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB) — skipping.")
else:
    with requests.get(URL, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        done = 0
        with open(ZIP_PATH, "wb") as f:
            for chunk in resp.iter_content(chunk_size=16 * 1024 * 1024):
                f.write(chunk)
                done += len(chunk)
                print(f"\r{done/1e9:.2f} GB", end="")
    print(f"\nSaved {ZIP_PATH}")

In [0]:
import zipfile
CSV_PATH = f"{RAW_DIR}/complaints.csv"

if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 1e9:
    print(f"Already extracted: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB) — skipping.")
else:
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)
        members = zf.namelist()
        extracted = os.path.join(RAW_DIR, members[0])
        if extracted != CSV_PATH:
            os.rename(extracted, CSV_PATH)
    print(f"Extracted to {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")

In [0]:
df = (spark.read
      .option("header", True)
      .option("multiLine", True)   
      .option("quote", '"')
      .option("escape", '"')       
      .csv(CSV_PATH))

df.printSchema()
display(df.limit(10))
n = df.count()
print(f"{n:,} rows")
assert n > 3_000_000,

In [0]:
from pyspark.sql import functions as F
(df
 .withColumn("_ingested_at", F.current_timestamp())
 .withColumn("_source_file", F.lit("complaints.csv.zip"))
 .write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")   
 .mode("overwrite")
 .saveAsTable("workspace.src.bronze_cfpb_complaints"))

In [0]:
import os, glob
HIST_DIR = "/Volumes/workspace/src/cfpb/history"
newest = max(glob.glob(f"{HIST_DIR}/complaints_*.csv"), key=os.path.getmtime)
print("removing possibly-partial:", newest)
os.remove(newest)

In [0]:
import os, calendar, time, requests

HIST_DIR = "/Volumes/workspace/src/cfpb/history"
os.makedirs(HIST_DIR, exist_ok=True)

BASE = ("https://www.consumerfinance.gov/data-research/consumer-complaints/"
        "search/api/v1/?format=csv&no_aggs=true&field=all")

def fetch(url, path, tries=6):
    tmp = path + ".part"
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, stream=True, timeout=300)
            if r.status_code >= 500:
                wait = 60 * attempt
                print(f"  {r.status_code} — waiting {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                    f.write(chunk)
            os.replace(tmp, path)          # atomic: only lands on full success
            return
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"  network drop ({type(e).__name__}) — retry {attempt}/{tries} in {30*attempt}s")
            time.sleep(30 * attempt)
    raise RuntimeError(f"Gave up: {url}")

for y in range(2013, 2019):
    for m in range(1, 13):
        ym = f"{y}-{m:02d}"
        path = f"{HIST_DIR}/complaints_{ym}.csv"
        if os.path.exists(path) and os.path.getsize(path) > 10_000:
            print(f"{ym}: exists, skipping")
            continue
        last_day = calendar.monthrange(y, m)[1]
        fetch(f"{BASE}&date_received_min={ym}-01&date_received_max={ym}-{last_day:02d}", path)
        print(f"{ym}: {os.path.getsize(path)/1e6:.1f} MB")
        time.sleep(10)

Silver layer

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.src.bronze_cfpb_complaints"
SILVER = "workspace.src.silver_cfpb_complaints"

RENAMES = {
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "narrative",
    "Company public response": "company_public_response",
    "Company": "company",
    "State": "state",
    "ZIP code": "zip_code",
    "Tags": "tags",
    "Submitted via": "submitted_via",
    "Date sent to company": "date_sent_to_company",
    "Company response to consumer": "company_response",
    "Timely response?": "timely_response",
    "Complaint ID": "complaint_id",
}

df = spark.table(BRONZE)
missing = [c for c in RENAMES if c not in df.columns]
assert not missing, f"Bronze is missing expected columns: {missing}"

for old, new in RENAMES.items():
    df = df.withColumnRenamed(old, new)

In [0]:
silver = (df
    .withColumn("complaint_id", F.col("complaint_id").cast("long"))
    .withColumn("date_received", F.to_date(F.col("date_received").cast("timestamp")))
    .withColumn("date_sent_to_company", F.to_date(F.col("date_sent_to_company").cast("timestamp")))
    .withColumn("timely_response", F.col("timely_response") == "Yes")
    .withColumn("has_narrative",
                F.col("narrative").isNotNull() & (F.length("narrative") > 0))
    .withColumn("year_month", F.date_format("date_received", "yyyy-MM"))
    .withColumn("product_family",
        F.when(F.col("product").rlike("(?i)personal loan|payday|consumer loan|installment"), "personal_loan")
         .when(F.col("product").rlike("(?i)credit card"), "credit_card")
         .when(F.col("product").rlike("(?i)mortgage"), "mortgage")
         .when(F.col("product").rlike("(?i)debt collection"), "debt_collection")
         .when(F.col("product").rlike("(?i)credit report"), "credit_reporting")
         .otherwise("other"))
    .dropDuplicates(["complaint_id"])
    .filter(F.col("complaint_id").isNotNull())
    .select(*RENAMES.values(), "has_narrative", "year_month", "product_family",
            "_ingested_at")
)

silver.write.mode("overwrite").saveAsTable(SILVER)

In [0]:
t = spark.table(SILVER)
print(f"{SILVER}: {t.count():,} rows, "
      f"{t.filter('has_narrative').count():,} with narrative")
display(t.groupBy("product_family").count().orderBy(F.desc("count")))

Gold

In [0]:
display(spark.sql("SHOW TABLES IN workspace.src"))

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/src/lendingclub/"))

In [0]:
from pyspark.sql import functions as F

LC_PATH = "/Volumes/workspace/src/lendingclub/accepted_2007_to_2018Q4.csv"

b = (spark.read
     .option("header", True)
     .option("multiLine", True)
     .option("quote", '"')
     .option("escape", '"')
     .csv(LC_PATH)
     .where(F.col("id").rlike("^[0-9]+$"))
     .withColumn("_ingested_at", F.current_timestamp())
     .withColumn("_source_file", F.lit("accepted_2007_to_2018Q4.csv")))

b.write.mode("overwrite").saveAsTable("workspace.src.bronze_lendingclub")
print(f"{spark.table('workspace.src.bronze_lendingclub').count():,} rows")
print(b.columns[:20])

In [0]:
display(spark.table("workspace.src.bronze_lendingclub").select("issue_d", "earliest_cr_line").limit(5))

In [0]:
from pyspark.sql import functions as F

b = spark.table("workspace.src.bronze_lendingclub")

def pct(c):
    return F.regexp_replace(F.col(c), "%", "").cast("double")

def to_int(c):
    return F.col(c).cast("double").cast("int")

s = (b
    .withColumn("loan_id", F.col("id").cast("long"))
    .withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))
    .withColumn("earliest_cr_date", F.to_date("earliest_cr_line", "MMM-yyyy"))
    .withColumn("loan_amnt", F.col("loan_amnt").cast("double"))
    .withColumn("funded_amnt", F.col("funded_amnt").cast("double"))
    .withColumn("term_months", F.regexp_extract("term", r"(\d+)", 1).cast("int"))
    .withColumn("int_rate", pct("int_rate"))
    .withColumn("installment", F.col("installment").cast("double"))
    .withColumn(
        "emp_length_years",
        F.when(F.col("emp_length").rlike(r"10\+"), F.lit(10))
        .when(F.col("emp_length").rlike("< 1"), F.lit(0))
        .otherwise(F.regexp_extract("emp_length", r"(\d+)", 1).cast("int")),
    )
    .withColumn("annual_inc", F.col("annual_inc").cast("double"))
    .withColumn("dti", F.col("dti").cast("double"))
    .withColumn("fico_low", to_int("fico_range_low"))
    .withColumn("fico_high", to_int("fico_range_high"))
    .withColumn("fico_avg", (F.col("fico_range_low").cast("double") + F.col("fico_range_high").cast("double")) / 2)
    .withColumn("revol_util", pct("revol_util"))
    .withColumn("revol_bal", F.col("revol_bal").cast("double"))
    .withColumn("open_acc", to_int("open_acc"))
    .withColumn("total_acc", to_int("total_acc"))
    .withColumn("delinq_2yrs", to_int("delinq_2yrs"))
    .withColumn("inq_last_6mths", to_int("inq_last_6mths"))
    .withColumn("pub_rec", to_int("pub_rec"))
    .withColumn("pub_rec_bankruptcies", to_int("pub_rec_bankruptcies"))
    .withColumn("mort_acc", to_int("mort_acc"))
    .withColumn("tot_cur_bal", F.col("tot_cur_bal").cast("double"))
    .withColumn("out_prncp", F.col("out_prncp").cast("double"))
    .withColumn("total_pymnt", F.col("total_pymnt").cast("double"))
    .withColumn("total_rec_prncp", F.col("total_rec_prncp").cast("double"))
    .withColumn("total_rec_int", F.col("total_rec_int").cast("double"))
    .withColumn("recoveries", F.col("recoveries").cast("double"))
    .withColumn("last_pymnt_amnt", F.col("last_pymnt_amnt").cast("double"))
    .withColumn("credit_age_years", F.round(F.months_between("issue_date", "earliest_cr_date") / 12, 1))
    .withColumn(
        "installment_to_income",
        F.when(F.col("annual_inc") > 0, F.round(F.col("installment") * 12 / F.col("annual_inc"), 4)),
    )
    .withColumn("is_resolved", F.when(F.lower("loan_status").rlike("fully paid|charged off|default"), 1).otherwise(0))
    .withColumn("is_default", F.when(F.lower("loan_status").rlike("charged off|default"), 1).otherwise(0))
    .withColumn("net_cash", F.col("total_pymnt") - F.col("funded_amnt"))
    .withColumn(
        "charge_off_loss",
        F.when(
            F.col("is_default") == 1,
            F.greatest(F.col("funded_amnt") - F.col("total_rec_prncp") - F.col("recoveries"), F.lit(0.0)),
        ).otherwise(F.lit(0.0)),
    )
)

keep = [
    "loan_id", "issue_date", "earliest_cr_date", "credit_age_years",
    "loan_amnt", "funded_amnt", "term_months", "int_rate", "installment",
    "grade", "sub_grade", "emp_length_years", "emp_title", "home_ownership",
    "annual_inc", "verification_status", "purpose", "title", "addr_state", "zip_code",
    "dti", "fico_low", "fico_high", "fico_avg", "revol_util", "revol_bal",
    "open_acc", "total_acc", "delinq_2yrs", "inq_last_6mths", "pub_rec",
    "pub_rec_bankruptcies", "mort_acc", "tot_cur_bal", "application_type",
    "installment_to_income",
    "loan_status", "is_resolved", "is_default",
    "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries",
    "last_pymnt_amnt", "net_cash", "charge_off_loss",
    "_ingested_at", "_source_file",
]

silver = (s.select(*keep)
    .dropDuplicates(["loan_id"])
    .filter("loan_amnt IS NOT NULL AND grade IS NOT NULL AND loan_status IS NOT NULL")
    .filter("loan_amnt > 0"))

silver.write.mode("overwrite").saveAsTable("workspace.src.silver_lendingclub")
print(f"{spark.table('workspace.src.silver_lendingclub').count():,} rows")

In [0]:
lc = spark.table("workspace.src.silver_lendingclub")

lc_seg = (lc
    .filter(F.col("issue_date").isNotNull())
    .withColumn("issue_month", F.date_format("issue_date", "yyyy-MM"))
    .groupBy(F.col("addr_state").alias("state"), "issue_month")
    .agg(F.count("*").alias("loans"),
         F.sum("is_default").alias("bad_loans"),
         F.sum("is_resolved").alias("resolved_loans"))
    .withColumn("default_rate",
        F.when(F.col("resolved_loans") > 0, F.col("bad_loans") / F.col("resolved_loans")))
)
lc_seg.write.mode("overwrite").saveAsTable("workspace.src.gold_lc_default_by_state_month")

In [0]:
cfpb_seg = (spark.table("workspace.src.silver_cfpb_complaints")
    .filter(F.col("product_family").isin("personal_loan", "credit_card", "debt_collection"))
    .groupBy("state", F.col("year_month").alias("issue_month"))
    .agg(F.count("*").alias("complaints"))
)
cfpb_seg.write.mode("overwrite").saveAsTable("workspace.src.gold_cfpb_complaints_by_state_month")

In [0]:
panel = (lc_seg.join(cfpb_seg, ["state", "issue_month"], "inner")
    .withColumn("complaints_per_1k_loans", F.col("complaints") / F.col("loans") * 1000)
    .filter(F.col("resolved_loans") >= 100)
)
panel.write.mode("overwrite").saveAsTable("workspace.src.gold_cfpb_lc_panel")

print("panel cells:", panel.count())
print("state×month r:", panel.stat.corr("complaints_per_1k_loans", "default_rate"))

state_level = (panel.groupBy("state")
    .agg((F.sum("bad_loans") / F.sum("resolved_loans")).alias("default_rate"),
         (F.sum("complaints") / F.sum("loans") * 1000).alias("complaints_per_1k_loans")))
print("state-level r:", state_level.stat.corr("complaints_per_1k_loans", "default_rate"))
display(state_level.orderBy(F.desc("default_rate")))

In [0]:
lcg = spark.table("workspace.src.gold_lc_default_by_state_month")
cfg = spark.table("workspace.src.gold_cfpb_complaints_by_state_month")

print("LC cells total:", lcg.count())
print("LC cells with resolved>=100:", lcg.filter("resolved_loans >= 100").count())
display(lcg.select(F.min("issue_month"), F.max("issue_month")))
display(lcg.orderBy(F.desc("resolved_loans")).limit(5))

print("CFPB cells total:", cfg.count())
display(cfg.select(F.min("issue_month"), F.max("issue_month")))
display(cfg.orderBy(F.desc("complaints")).limit(5))

In [0]:
sc = spark.table("workspace.src.silver_cfpb_complaints")

print("null year_month:", sc.filter("year_month IS NULL").count())
display(sc.select(F.min("date_received"), F.max("date_received")))

# complaints per year, gold families vs everything
fam = sc.filter(F.col("product_family").isin("personal_loan","credit_card","debt_collection"))
display(fam.groupBy(F.substring("year_month",1,4).alias("yr")).count().orderBy("yr"))

# what raw product names exist pre-2019? (catches taxonomy the regex missed)
display(sc.filter("year_month < '2019-01'")
          .groupBy("product", "product_family").count()
          .orderBy(F.desc("count")).limit(20))

In [0]:
b = spark.table("workspace.src.bronze_cfpb_complaints")
s = spark.table("workspace.src.silver_cfpb_complaints")

print("bronze:", b.count())
print("silver:", s.count())
print("bronze null Complaint ID:", b.filter(F.col("`Complaint ID`").isNull()).count())
display(b.groupBy(F.substring("`Date received`", 1, 4).alias("yr")).count().orderBy("yr"))

In [0]:
import zipfile
z = zipfile.ZipFile("/Volumes/workspace/src/cfpb/raw/complaints.csv.zip")
print([(i.filename, round(i.file_size/1e9, 2)) for i in z.infolist()])

In [0]:
df = (spark.table("workspace.src.bronze_cfpb_complaints")
      .unionByName(spark.table("workspace.src.bronze_cfpb_history"), allowMissingColumns=True))

In [0]:
from pyspark.sql import functions as F
h = spark.table("workspace.src.bronze_cfpb")
print(f"{h.count():,} rows")
display(h.groupBy(F.substring("`Date received`", 1, 4).alias("yr")).count().orderBy("yr"))